In [1]:
import pandas as pd
import requests
import numpy as np
from bs4 import BeautifulSoup
import yfinance
import os
import pygsheets
from decimal import Decimal, ROUND_HALF_UP
import locale
from datetime import datetime, timedelta
import json
import datawrapper
import time
import io

# Set the locale to English (United States)
locale.setlocale(locale.LC_TIME, 'en_US.UTF-8')
#authorization
if "PYGSHEET" in os.environ:

    AUTH_DATA = os.environ["PYGSHEET"]
    whatever_json=json.loads(AUTH_DATA, strict=False)
    credentials = whatever_json

    gc = pygsheets.authorize(service_account_json=(json.dumps(whatever_json))
    )
else:

    gc = pygsheets.authorize(service_file='/Users/halukamaier-borst/Documents/trumptracker-459020-5b428916efa8.json')



In [2]:

# Get today's date
today = datetime.now()

# Get date 365 days ago
date_365_days_ago = today - timedelta(days=365)

# Format as string if needed
date_365_days_ago_str = date_365_days_ago.strftime("%Y-%m-%d")

print(date_365_days_ago_str)
today

2025-01-28


datetime.datetime(2026, 1, 28, 10, 54, 20, 529295)

In [3]:

shanghaicomposite = yfinance.download (tickers = "000001.SS", start = date_365_days_ago_str, 
                              interval = "1d")
shanghaicomposite

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,000001.SS,000001.SS,000001.SS,000001.SS,000001.SS
Date,,,,,
2025-02-05,3229.488037,3271.021973,3220.279053,3270.159912,406400
2025-02-06,3270.658936,3271.427002,3225.845947,3226.574951,460800
2025-02-07,3303.666992,3324.427979,3262.509033,3269.227051,600600
2025-02-10,3322.169922,3325.355957,3299.242920,3308.040039,546600
2025-02-11,3318.056885,3324.173096,3304.444092,3323.668945,511900
...,...,...,...,...,...
2026-01-22,4122.576172,4140.837891,4109.922852,4126.071777,709700
2026-01-23,4136.164062,4143.750000,4120.204102,4130.106934,782100


In [4]:

dax40 = yfinance.download (tickers = "^GDAXI", start = date_365_days_ago_str,
                              interval = "1d")
dax40

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^GDAXI,^GDAXI,^GDAXI,^GDAXI,^GDAXI
Date,,,,,
2025-01-28,21430.580078,21475.900391,21296.330078,21374.289062,68020300
2025-01-29,21637.529297,21671.589844,21475.640625,21511.470703,69568800
2025-01-30,21727.199219,21732.050781,21650.779297,21676.220703,81446400
2025-01-31,21732.050781,21800.519531,21689.289062,21771.490234,70233900
2025-02-03,21428.240234,21462.789062,21252.710938,21301.529297,80512800
...,...,...,...,...,...
2026-01-22,24856.470703,24937.080078,24772.539062,24877.630859,68349500
2026-01-23,24900.710938,24930.150391,24796.929688,24819.939453,51854400


In [5]:
sandp500 = yfinance.download (tickers = "^GSPC", start = date_365_days_ago_str, 
                              interval = "1d")
sandp500

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
2025-01-28,6067.700195,6074.540039,5994.629883,6026.970215,4473640000
2025-01-29,6039.310059,6062.830078,6012.959961,6057.700195,4115310000
2025-01-30,6071.169922,6086.640137,6027.459961,6050.750000,4634620000
2025-01-31,6040.529785,6120.910156,6030.930176,6096.790039,4843770000
2025-02-03,5994.569824,6022.129883,5923.930176,5969.649902,4797210000
...,...,...,...,...,...
2026-01-21,6875.620117,6910.390137,6804.959961,6810.709961,5835520000
2026-01-22,6913.350098,6934.750000,6893.620117,6914.439941,5307580000


In [6]:
def reduce_index(df,indexname):
    index_red=pd.DataFrame()
    df["Date"]=df.index
    index_red["Date"]=df["Date"]

    index_red[indexname]=df[df.columns[0]]

    index_red=index_red.reset_index(drop=True)
    return index_red


In [7]:
sandp500_red=reduce_index(sandp500,"S&P500")
dax40_red=reduce_index(dax40,"DAX40")
shanghaicomposite_red=reduce_index(shanghaicomposite,"SHANGHAI")
shanghaicomposite_red

,Date,SHANGHAI
0,2025-02-05,3229.488037
1,2025-02-06,3270.658936
2,2025-02-07,3303.666992
3,2025-02-10,3322.169922
4,2025-02-11,3318.056885
...,...,...
238,2026-01-22,4122.576172
239,2026-01-23,4136.164062
240,2026-01-26,4132.604980
241,2026-01-27,4139.903809


In [8]:
sandp500_dax40=pd.merge(sandp500_red, dax40_red, on="Date", how="outer")
sandp500_dax40=sandp500_dax40.fillna(method="ffill")
sandp500_dax40

/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_31652/2964685221.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  sandp500_dax40=sandp500_dax40.fillna(method="ffill")


,Date,S&P500,DAX40
0,2025-01-28,6067.700195,21430.580078
1,2025-01-29,6039.310059,21637.529297
2,2025-01-30,6071.169922,21727.199219
3,2025-01-31,6040.529785,21732.050781
4,2025-02-03,5994.569824,21428.240234
...,...,...,...
254,2026-01-22,6913.350098,24856.470703
255,2026-01-23,6915.609863,24900.710938
256,2026-01-26,6950.229980,24933.080078
257,2026-01-27,6978.600098,24894.439453


In [9]:
sandp500_dax40_shanghai=pd.merge(sandp500_dax40, shanghaicomposite_red, on="Date", how="outer")
sandp500_dax40_shanghai=sandp500_dax40_shanghai.fillna(method="ffill")
sandp500_dax40_shanghai

/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_31652/2476299639.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  sandp500_dax40_shanghai=sandp500_dax40_shanghai.fillna(method="ffill")


,Date,S&P500,DAX40,SHANGHAI
0,2025-01-28,6067.700195,21430.580078,NaN
1,2025-01-29,6039.310059,21637.529297,NaN
2,2025-01-30,6071.169922,21727.199219,NaN
3,2025-01-31,6040.529785,21732.050781,NaN
4,2025-02-03,5994.569824,21428.240234,NaN
...,...,...,...,...
256,2026-01-22,6913.350098,24856.470703,4122.576172
257,2026-01-23,6915.609863,24900.710938,4136.164062
258,2026-01-26,6950.229980,24933.080078,4132.604980
259,2026-01-27,6978.600098,24894.439453,4139.903809


In [10]:
now = datetime.now()
today=now.strftime("%Y-%m-%d")
since_election=sandp500_dax40_shanghai[sandp500_dax40_shanghai["Date"]>=date_365_days_ago_str]
since_election_without_today=since_election[since_election["Date"]!=today]
since_election_without_today

,Date,S&P500,DAX40,SHANGHAI
0,2025-01-28,6067.700195,21430.580078,NaN
1,2025-01-29,6039.310059,21637.529297,NaN
2,2025-01-30,6071.169922,21727.199219,NaN
3,2025-01-31,6040.529785,21732.050781,NaN
4,2025-02-03,5994.569824,21428.240234,NaN
...,...,...,...,...
255,2026-01-21,6875.620117,24560.980469,4116.939941
256,2026-01-22,6913.350098,24856.470703,4122.576172
257,2026-01-23,6915.609863,24900.710938,4136.164062
258,2026-01-26,6950.229980,24933.080078,4132.604980


In [11]:
for col in since_election_without_today.columns[1:]:
    since_election_without_today[col]= since_election_without_today[col].apply(lambda x: Decimal(str(x)))

    # Example: Perform precise rounding on the Decimal column
    since_election_without_today[col]= since_election_without_today[col].apply(lambda x: x.quantize(Decimal('1'), rounding=ROUND_HALF_UP))
since_election_without_today

/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_31652/3670032238.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  since_election_without_today[col]= since_election_without_today[col].apply(lambda x: Decimal(str(x)))
/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_31652/3670032238.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  since_election_without_today[col]= since_election_without_today[col].apply(lambda x: x.quantize(Decimal('1'), rounding=ROUND_HALF_UP))


,Date,S&P500,DAX40,SHANGHAI
0,2025-01-28,6068,21431,NaN
1,2025-01-29,6039,21638,NaN
2,2025-01-30,6071,21727,NaN
3,2025-01-31,6041,21732,NaN
4,2025-02-03,5995,21428,NaN
...,...,...,...,...
255,2026-01-21,6876,24561,4117
256,2026-01-22,6913,24856,4123
257,2026-01-23,6916,24901,4136
258,2026-01-26,6950,24933,4133


In [12]:
since_election_without_today[since_election_without_today["Date"]=="2025-05-01"]

,Date,S&P500,DAX40,SHANGHAI
67,2025-05-01,5604,22497,3279


In [13]:
def retrieve_from_investing(index):
    index_to_shortform_dict={"S&P500":"us-spx-500","DAX40":"germany-30",	"SHANGHAI":"shanghai-composite"}
    now = datetime.now()
    formatted_date = now.strftime("%m/%d/%Y")
    index_shortform=index_to_shortform_dict[index]

    time.sleep(10)

    url=f"https://www.investing.com/indices/{index_shortform}-historical-data#:~:text=Daily-,Download,-{formatted_date}"
    print(url)
    response = requests.get(url)


    # Check if the request was successful
    if response.status_code == 200:
        # Parse the HTML content of the page
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find the first table on the page
        table = soup.find('table', class_="freeze-column-w-1 w-full overflow-x-auto text-xs leading-4")

        # Extract rows from the table
        rows = table.find_all('tr')

        # Extract data from each row
        table_data = []
        for row in rows:
            cells = row.find_all(['td', 'th'])  # Get all cells (both <td> and <th>)
            cell_data = [cell.get_text(strip=True) for cell in cells]  # Extract text
            table_data.append(cell_data)

        # Return the extracted table data
        return(table_data)
    else:
        print(f"Failed to retrieve the page. Status code: {response.status_code}")

In [14]:
df_to_check=since_election_without_today
columns_to_round = ['S&P500','DAX40','SHANGHAI']

# Apply Decimal rounding to the specified columns
df_to_check[columns_to_round] = df_to_check[columns_to_round].applymap(lambda x: Decimal(str(x)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))

for index_name in since_election_without_today.columns[1:]:
    print(index_name)
    array=retrieve_from_investing(index_name)
    crosscheck_data=pd.DataFrame(array)
    crosscheck_data.replace("",np.nan,inplace=True)

    crosscheck_data=crosscheck_data.dropna(how="all")
    crosscheck_data[0]=pd.to_datetime(crosscheck_data[0], format="%b %d, %Y", errors="coerce")
    crosscheck_data_red=crosscheck_data[[0,1]]
    crosscheck_data_red=crosscheck_data_red.dropna(subset=[0])
    crosscheck_data_red.columns=["Date",index_name+"check"]
    crosscheck_data_red=crosscheck_data_red.sort_values(by="Date", ascending=True)
    crosscheck_data_red[index_name+"check"]=crosscheck_data_red[index_name+"check"].str.replace(",","")
    crosscheck_data_red[index_name+"check"]=crosscheck_data_red[index_name+"check"].astype(float)

    crosscheck_data_red[index_name+"check"]= crosscheck_data_red[index_name+"check"].apply(lambda x: Decimal(str(x)))

# Example: Perform precise rounding on the Decimal column
    crosscheck_data_red[index_name+"check"] = crosscheck_data_red[index_name+"check"].apply(lambda x: x.quantize(Decimal('1'), rounding=ROUND_HALF_UP))
    print("crosscheckdata")
    print(crosscheck_data_red.tail(5))
    print("df_to_check")

    print(df_to_check.tail(5))

#    df_to_check=pd.merge(df_to_check,crosscheck_data_red, on="Date", how="outer", suffixes=["_raw","_crosscheck"])

    df_to_check=pd.merge(df_to_check,crosscheck_data_red, on="Date", how="outer", suffixes=["_raw","_crosscheck"])
df_to_check=df_to_check.fillna(method="ffill")
df_to_check


/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_31652/1710883695.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_to_check[columns_to_round] = df_to_check[columns_to_round].applymap(lambda x: Decimal(str(x)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))
/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_31652/1710883695.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_to_check[columns_to_round] = df_to_check[columns_to_round].applymap(lambda x: Decimal(str(x)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))


S&P500
https://www.investing.com/indices/us-spx-500-historical-data#:~:text=Daily-,Download,-01/28/2026


/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_31652/1710883695.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  crosscheck_data.replace("",np.nan,inplace=True)


crosscheckdata
        Date S&P500check
5 2026-01-21        6876
4 2026-01-22        6913
3 2026-01-23        6916
2 2026-01-26        6950
1 2026-01-27        6979
df_to_check
          Date S&P500  DAX40 SHANGHAI
255 2026-01-21   6876  24561     4117
256 2026-01-22   6913  24856     4123
257 2026-01-23   6916  24901     4136
258 2026-01-26   6950  24933     4133
259 2026-01-27   6979  24894     4140
DAX40
https://www.investing.com/indices/germany-30-historical-data#:~:text=Daily-,Download,-01/28/2026
crosscheckdata
        Date DAX40check
5 2026-01-22      24856
4 2026-01-23      24901
3 2026-01-26      24933
2 2026-01-27      24894
1 2026-01-28      24831
df_to_check
          Date S&P500  DAX40 SHANGHAI S&P500check
255 2026-01-21   6876  24561     4117        6876
256 2026-01-22   6913  24856     4123        6913
257 2026-01-23   6916  24901     4136        6916
258 2026-01-26   6950  24933     4133        6950
259 2026-01-27   6979  24894     4140        6979
SHANGHAI
https://www.

/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_31652/1710883695.py:35: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_to_check=df_to_check.fillna(method="ffill")


,Date,S&P500,DAX40,SHANGHAI,S&P500check,DAX40check,SHANGHAIcheck
0,2025-01-28,6068,21431,NaN,NaN,NaN,NaN
1,2025-01-29,6039,21638,NaN,NaN,NaN,NaN
2,2025-01-30,6071,21727,NaN,NaN,NaN,NaN
3,2025-01-31,6041,21732,NaN,NaN,NaN,NaN
4,2025-02-03,5995,21428,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
256,2026-01-22,6913,24856,4123,6913,24856,4123
257,2026-01-23,6916,24901,4136,6916,24901,4136
258,2026-01-26,6950,24933,4133,6950,24933,4133
259,2026-01-27,6979,24894,4140,6979,24894,4140


In [15]:
for index_name in since_election_without_today.columns[1:]:

    df_to_check[index_name+"overlap"]=df_to_check[index_name]/df_to_check[index_name+"check"]
df_to_check

,Date,S&P500,DAX40,SHANGHAI,S&P500check,DAX40check,SHANGHAIcheck,S&P500overlap,DAX40overlap,SHANGHAIoverlap
0,2025-01-28,6068,21431,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-29,6039,21638,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-01-30,6071,21727,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-01-31,6041,21732,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-02-03,5995,21428,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
256,2026-01-22,6913,24856,4123,6913,24856,4123,1,1,1
257,2026-01-23,6916,24901,4136,6916,24901,4136,1,1,1
258,2026-01-26,6950,24933,4133,6950,24933,4133,1,1,1
259,2026-01-27,6979,24894,4140,6979,24894,4140,1,1,1


In [16]:
overlap_cols=df_to_check.columns[df_to_check.columns.str.contains("overlap")]
overlap_cols

Index(['S&P500overlap', 'DAX40overlap', 'SHANGHAIoverlap'], dtype='object')

In [17]:
(df_to_check[overlap_cols]==1).all(axis=1)

0      False
1      False
2      False
3      False
4      False
       ...  
256     True
257     True
258     True
259     True
260    False
Length: 261, dtype: bool

In [18]:
df_to_check["all_equal"]=(df_to_check[overlap_cols]==1).all(axis=1)
df_to_check

,Date,S&P500,DAX40,SHANGHAI,S&P500check,DAX40check,SHANGHAIcheck,S&P500overlap,DAX40overlap,SHANGHAIoverlap,all_equal
0,2025-01-28,6068,21431,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,2025-01-29,6039,21638,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,2025-01-30,6071,21727,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
3,2025-01-31,6041,21732,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4,2025-02-03,5995,21428,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...
256,2026-01-22,6913,24856,4123,6913,24856,4123,1,1,1,True
257,2026-01-23,6916,24901,4136,6916,24901,4136,1,1,1,True
258,2026-01-26,6950,24933,4133,6950,24933,4133,1,1,1,True
259,2026-01-27,6979,24894,4140,6979,24894,4140,1,1,1,True


In [19]:

checked_df=df_to_check[df_to_check["all_equal"]==True]
checked_df

,Date,S&P500,DAX40,SHANGHAI,S&P500check,DAX40check,SHANGHAIcheck,S&P500overlap,DAX40overlap,SHANGHAIoverlap,all_equal
239,2025-12-29,6906,24351,3965,6906,24351,3965,1,1,1,True
240,2025-12-30,6896,24490,3965,6896,24490,3965,1,1,1,True
241,2025-12-31,6846,24490,3969,6846,24490,3969,1,1,1,True
242,2026-01-02,6858,24539,3969,6858,24539,3969,1,1,1,True
243,2026-01-05,6902,24869,4023,6902,24869,4023,1,1,1,True
244,2026-01-06,6945,24892,4084,6945,24892,4084,1,1,1,True
245,2026-01-07,6921,25122,4086,6921,25122,4086,1,1,1,True
246,2026-01-08,6921,25127,4083,6921,25127,4083,1,1,1,True
247,2026-01-09,6966,25262,4120,6966,25262,4120,1,1,1,True
248,2026-01-12,6977,25405,4165,6977,25405,4165,1,1,1,True


In [20]:
checked_df_red=checked_df[checked_df.columns[:4]]
checked_df_red["Date"]=checked_df_red["Date"].dt.strftime("%Y-%m-%d")
checked_df_red

/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_31652/3470133631.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  checked_df_red["Date"]=checked_df_red["Date"].dt.strftime("%Y-%m-%d")


,Date,S&P500,DAX40,SHANGHAI
239,2025-12-29,6906,24351,3965
240,2025-12-30,6896,24490,3965
241,2025-12-31,6846,24490,3969
242,2026-01-02,6858,24539,3969
243,2026-01-05,6902,24869,4023
244,2026-01-06,6945,24892,4084
245,2026-01-07,6921,25122,4086
246,2026-01-08,6921,25127,4083
247,2026-01-09,6966,25262,4120
248,2026-01-12,6977,25405,4165


In [21]:
#Open the google spreadsheet
sh = gc.open('TrumptrackerRaw')
sheet_name = "RawIndices"  # Replace with the name of the sheet you want to select
wks = sh.worksheet_by_title(sheet_name)


#Get the data from the Sheet into python as DF
read_indices = wks.get_as_df()
read_indices_red=read_indices[read_indices.columns[:4]]
read_indices_red

,Date,S&P500,DAX40,SHANGHAI
0,2025-01-29,6039,21638,3251
1,2025-01-30,6071,21727,3251
2,2025-01-31,6041,21732,3251
3,2025-02-03,5995,21428,3251
4,2025-02-04,6038,21506,3251
...,...,...,...,...
253,2026-01-21,6876,24561,4117
254,2026-01-22,6876,24561,4117
255,2026-01-23,6916,24901,4136
256,2026-01-26,6950,24933,4133


In [27]:
read_plus_new_checked=pd.concat([read_indices_red,checked_df_red], axis=0, ignore_index=True)

read_plus_new_checked.drop_duplicates(subset=["Date"], keep="first", inplace=True)
read_plus_new_checked.reset_index(drop=True, inplace=True)
read_plus_new_checked

,Date,S&P500,DAX40,SHANGHAI
0,2025-01-29,6039,21638,3251
1,2025-01-30,6071,21727,3251
2,2025-01-31,6041,21732,3251
3,2025-02-03,5995,21428,3251
4,2025-02-04,6038,21506,3251
...,...,...,...,...
253,2026-01-21,6876,24561,4117
254,2026-01-22,6876,24561,4117
255,2026-01-23,6916,24901,4136
256,2026-01-26,6950,24933,4133


In [28]:
read_plus_new_checked["Date"]=pd.to_datetime(read_plus_new_checked["Date"])
read_plus_new_checked=read_plus_new_checked[read_plus_new_checked["Date"]>date_365_days_ago]
read_plus_new_checked

,Date,S&P500,DAX40,SHANGHAI
0,2025-01-29,6039,21638,3251
1,2025-01-30,6071,21727,3251
2,2025-01-31,6041,21732,3251
3,2025-02-03,5995,21428,3251
4,2025-02-04,6038,21506,3251
...,...,...,...,...
253,2026-01-21,6876,24561,4117
254,2026-01-22,6876,24561,4117
255,2026-01-23,6916,24901,4136
256,2026-01-26,6950,24933,4133


In [29]:
def converter_perc(index_name):
    df_perc=pd.DataFrame()
    df_perc["Date"]=read_plus_new_checked["Date"]
    col_important=read_plus_new_checked[index_name]
    base_value=col_important.iloc[0]
    df_perc[index_name+"_perc"]=(col_important-base_value)*100/base_value
    df_perc[index_name+"_perc"] = df_perc[index_name+"_perc"].apply(lambda x: float(Decimal(str(x)).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)))

    return(df_perc)

In [30]:
for col in read_plus_new_checked.columns[1:]:
    df_perc=converter_perc(col)
    read_plus_new_checked=pd.merge(read_plus_new_checked,df_perc,on="Date",how="outer",suffixes=["_abs","_perc"])
read_plus_new_checked=read_plus_new_checked.sort_values(by="Date", ascending=True)
read_plus_new_checked

,Date,S&P500,DAX40,SHANGHAI,S&P500_perc,DAX40_perc,SHANGHAI_perc
0,2025-01-29,6039,21638,3251,0.00,0.00,0.00
1,2025-01-30,6071,21727,3251,0.53,0.41,0.00
2,2025-01-31,6041,21732,3251,0.03,0.43,0.00
3,2025-02-03,5995,21428,3251,-0.73,-0.97,0.00
4,2025-02-04,6038,21506,3251,-0.02,-0.61,0.00
...,...,...,...,...,...,...,...
253,2026-01-21,6876,24561,4117,13.86,13.51,26.64
254,2026-01-22,6876,24561,4117,13.86,13.51,26.64
255,2026-01-23,6916,24901,4136,14.52,15.08,27.22
256,2026-01-26,6950,24933,4133,15.09,15.23,27.13


In [31]:



#open the google spreadsheet (where 'PY to Gsheet Test' is the name of my sheet)
sh = gc.open('TrumptrackerRaw')

#select the first sheet 
wks = sh[0]

wks.set_dataframe(read_plus_new_checked,(0,0))


/Users/halukamaier-borst/Documents/CodeRelated/HBDataviz_HalukaOnly/2025/08_Trumptracker/.venv/lib/python3.12/site-packages/pygsheets/worksheet.py:1431: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(nan)


In [32]:
# Get the current year
current_year = datetime.now().year
print(current_year)

2026


In [33]:
df=pd.DataFrame()
for year in range(2024,current_year+1):
    print(year)
    url=f"https://home.treasury.gov/resource-center/data-chart-center/interest-rates/daily-treasury-rates.csv/{year}/all?type=daily_treasury_yield_curve&field_tdr_date_value={year}&page&_format=csv"
    response = requests.get(url)

    file_object = io.StringIO(response.content.decode('utf-8'))
    data=pd.read_csv(file_object)
    df=pd.concat([df,data],axis=0)
df

2024
2025
2026


,Date,1 Mo,2 Mo,3 Mo,4 Mo,6 Mo,1 Yr,2 Yr,3 Yr,5 Yr,7 Yr,10 Yr,20 Yr,30 Yr,1.5 Month
0,12/31/2024,4.40,4.39,4.37,4.32,4.24,4.16,4.25,4.27,4.38,4.48,4.58,4.86,4.78,NaN
1,12/30/2024,4.43,4.42,4.37,4.33,4.25,4.17,4.24,4.29,4.37,4.46,4.55,4.84,4.77,NaN
2,12/27/2024,4.44,4.43,4.31,4.35,4.29,4.20,4.31,4.36,4.45,4.53,4.62,4.89,4.82,NaN
3,12/26/2024,4.45,4.45,4.35,4.37,4.31,4.23,4.30,4.35,4.42,4.49,4.58,4.83,4.76,NaN
4,12/24/2024,4.44,4.44,4.40,4.38,4.30,4.24,4.29,4.36,4.43,4.52,4.59,4.84,4.76,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12,01/08/2026,3.70,3.63,3.62,3.60,3.56,3.48,3.49,3.56,3.74,3.95,4.19,4.79,4.85,3.69
13,01/07/2026,3.70,3.60,3.62,3.60,3.56,3.48,3.47,3.53,3.70,3.91,4.15,4.76,4.82,3.69
14,01/06/2026,3.70,3.61,3.63,3.58,3.56,3.48,3.47,3.54,3.72,3.93,4.18,4.80,4.86,3.70
15,01/05/2026,3.71,3.64,3.64,3.61,3.57,3.47,3.46,3.53,3.71,3.92,4.17,4.79,4.85,3.68


In [29]:
df["Date"]=pd.to_datetime(df["Date"])
since_election_yield=df[df["Date"]>=date_365_days_ago]
since_election_yield=since_election_yield[["Date","10 Yr"]]
since_election_yield=since_election_yield.sort_values(by="Date", ascending=True)
since_election_yield

,Date,10 Yr
13,2024-12-11,4.26
12,2024-12-12,4.32
11,2024-12-13,4.40
10,2024-12-16,4.39
9,2024-12-17,4.40
...,...,...
4,2025-12-03,4.06
3,2025-12-04,4.11
2,2025-12-05,4.14
1,2025-12-08,4.17


In [30]:



#open the google spreadsheet (where 'PY to Gsheet Test' is the name of my sheet)
sh = gc.open('TrumptrackerRaw')

#select the first sheet 
sheet_name = "Yield"  # Replace with the name of the sheet you want to select
wks = sh.worksheet_by_title(sheet_name)


wks.set_dataframe(since_election_yield,(0,0))


In [31]:
if "DWKEY" in os.environ:

    DWKEY = os.environ["DWKEY"]
    print(len(DWKEY))


else:
    data=pd.read_csv("/Users/halukamaier-borst/Documents/Keys.csv")
    DWKEY=data.APIKey.iloc[1]

In [32]:
dw_connection = datawrapper.Datawrapper(access_token = DWKEY)

allcharts=dw_connection.get_charts(limit=500)

dw_connection.publish_chart(chart_id="50HqS")

{'data': {'publicId': '50HqS',
  'language': 'de-DE',
  'theme': 'handelsblatt',
  'id': '50HqS',
  'type': 'd3-lines',
  'title': 'So haben sich die Märkte in den letzten 365 Tagen verändert',
  'lastEditStep': 5,
  'publishedAt': '2025-12-10T08:48:11.992Z',
  'publicUrl': 'https://datawrapper.dwcdn.net/50HqS/827/',
  'publicVersion': 827,
  'deleted': False,
  'forkable': False,
  'isFork': False,
  'metadata': {'data': {'changes': [],
    'transpose': False,
    'vertical-header': True,
    'horizontal-header': True,
    'column-format': {'DAX40': {'type': 'auto', 'ignore': True},
     'S&P500': {'type': 'auto', 'ignore': True},
     'SHANGHAI': {'type': 'auto', 'ignore': True},
     'DAX40_abs': {'type': 'auto', 'ignore': True},
     'DAX40_perc': {'type': 'auto',
      'number-append': ' %',
      'number-prepend': ''},
     'S&P500_abs': {'type': 'auto', 'ignore': True},
     'S&P500_perc': {'type': 'auto',
      'number-append': ' %',
      'number-prepend': ''},
     'SHANGHAI_